# Sistema de recomendación item-item por canastas

Esta libreta utiliza el dataset de interacciones `dataset_recomendacion_interacciones.csv` y el catálogo `dataset_productos.csv`. Cada pedido `Entregado` representa una canasta (`presencia = 1`), se cuentan productos presentes y pares comprados juntos, y se calcula la **similitud coseno**.


## 1. Dependencias e importación de librerías


In [ ]:
import os
from collections import Counter
from itertools import combinations
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Cargar el dataset


In [ ]:
# Buscar archivos CSV
ruta_interacciones = next((p for p in [
    Path.cwd() / "outputs" / "dataset_recomendacion_interacciones.csv",
    Path.cwd().parent / "outputs" / "dataset_recomendacion_interacciones.csv",
    Path.cwd() / "dataset_recomendacion_interacciones.csv"
] if p.exists()), "outputs/dataset_recomendacion_interacciones.csv")

ruta_productos = next((p for p in [
    Path.cwd() / "outputs" / "dataset_productos.csv",
    Path.cwd().parent / "outputs" / "dataset_productos.csv",
    Path.cwd() / "dataset_productos.csv"
] if p.exists()), "outputs/dataset_productos.csv")

df = pd.read_csv(ruta_interacciones)
catalogo = pd.read_csv(ruta_productos)

print("Total de presencias:", len(df))
df.head(10)


## 3. Extraer Canastas y Frecuencias


In [ ]:
# Agrupar por pedido para formar las canastas
canastas = []
for pedido_id, grupo in df.groupby("pedido_id"):
    productos = sorted(grupo["producto_id"].astype(str).unique().tolist())
    if productos:
        canastas.append({"pedido": pedido_id, "productos": productos})

# Contar frecuencias individuales y coocurrencias de pares
frecuencia = Counter()
coocurrencia = Counter()
for canasta in canastas:
    productos = canasta["productos"]
    frecuencia.update(productos)
    coocurrencia.update(combinations(productos, 2))

print("Canastas analizadas:", len(canastas))
print("Productos distintos:", len(frecuencia))
print("Pares con coocurrencia:", len(coocurrencia))


## 4. Cálculo de Similitud Coseno


In [ ]:
filas_similitud = []
for (producto_a, producto_b), juntos in coocurrencia.items():
    frecuencia_a = frecuencia[producto_a]
    frecuencia_b = frecuencia[producto_b]
    similitud = juntos / ((frecuencia_a * frecuencia_b) ** 0.5)
    filas_similitud.append({
        "producto_A": producto_a, "producto_B": producto_b,
        "coocurrencias": juntos, "frecuencia_A": frecuencia_a,
        "frecuencia_B": frecuencia_b, "similitud": round(similitud, 4)
    })

pares = pd.DataFrame(filas_similitud).sort_values("similitud", ascending=False)
pares.head(10)


## 5. Función para Recomendar Productos en Carrito


In [ ]:
catalogo["producto_id"] = catalogo["producto_id"].astype(str)
catalogo_idx = catalogo.set_index("producto_id")

def similitud_par(producto_a, producto_b):
    clave = tuple(sorted((producto_a, producto_b)))
    juntos = coocurrencia.get(clave, 0)
    if not juntos:
        return 0
    return juntos / ((frecuencia[producto_a] * frecuencia[producto_b]) ** 0.5)

def recomendar_carrito(producto_ids, top_n=6):
    semillas = list(dict.fromkeys(str(p) for p in producto_ids if str(p) in frecuencia))
    puntuaciones = Counter()
    for semilla in semillas:
        for candidato in frecuencia:
            if candidato in semillas:
                continue
            puntuaciones[candidato] += similitud_par(semilla, candidato)
    resultado = pd.DataFrame([
        {"producto_id": producto, "puntuacion": round(puntuacion, 4)}
        for producto, puntuacion in puntuaciones.items() if puntuacion > 0
    ]).set_index("producto_id").join(catalogo_idx, how="left")
    resultado = resultado[(resultado["activo"] != 0) & (resultado["stock"] > 0)]
    return resultado.sort_values("puntuacion", ascending=False)[
        ["nombre", "marca", "familia", "precioNormal", "puntuacion"]
    ].head(top_n)

carrito_ejemplo = list(frecuencia.keys())[:3]
print("Carrito de ejemplo:", catalogo_idx.reindex(carrito_ejemplo)["nombre"].dropna().tolist())
recomendar_carrito(carrito_ejemplo)


## 6. Visualización con Mapa de Calor


In [ ]:
muestra = list(frecuencia.keys())[:15]
matriz_similitud = pd.DataFrame(0.0, index=muestra, columns=muestra)
for producto_a in muestra:
    matriz_similitud.loc[producto_a, producto_a] = 1.0
    for producto_b in muestra:
        if producto_a != producto_b:
            matriz_similitud.loc[producto_a, producto_b] = similitud_par(producto_a, producto_b)

plt.figure(figsize=(12, 8))
sns.heatmap(matriz_similitud, cmap="YlGnBu", vmin=0, vmax=1)
plt.title("Similitud Coseno por Coocurrencia entre Productos")
plt.tight_layout()
plt.show()
